# Optional real-time CNN control / YOLOX-S

Anchor-free one-stage control. The notebook intentionally requires the official YOLOX repository; it never substitutes a differently licensed package.

License and exact weight provenance are recorded in `LICENSES.md` and each run manifest. Approximate GPU requirements depend strongly on resolution, batch size, AMP, and the active Colab GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GITHUB_USERNAME = "Harryphan72007"
GITHUB_REPOSITORY = "aerial-object-detection-benchmark"
DEFAULT_BRANCH = "main"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}.git"
REPO_DIR = f"/content/{GITHUB_REPOSITORY}"
DRIVE_ROOT = "/content/drive/MyDrive/visdrone_architecture_benchmark"
assert GITHUB_USERNAME != "<MY_GITHUB_USERNAME>"
import os, sys, subprocess
if os.path.isdir(REPO_DIR) and not os.path.isdir(os.path.join(REPO_DIR, '.git')): raise RuntimeError(f'Existing non-Git directory: {REPO_DIR}')
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
if not os.path.isdir(os.path.join(REPO_DIR, '.git')): subprocess.run(['git', 'clone', '--branch', DEFAULT_BRANCH, REPO_URL, REPO_DIR], check=True)
from src.colab_setup import clone_or_update_repository, install_project, initialize_drive_directories, load_project_config, validate_drive_writable
clone_or_update_repository(REPO_URL, REPO_DIR, DEFAULT_BRANCH)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
install_project(REPO_DIR)
config = load_project_config('project_config.yaml')
paths = initialize_drive_directories(DRIVE_ROOT)
validate_drive_writable(DRIVE_ROOT)
print(subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip())
print(subprocess.run(['git', 'status', '--short'], capture_output=True, text=True, check=True).stdout)


In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

## Editable experiment configuration

In [ ]:
MODEL_ID = "yolox_s"
DATASET_TRACK = "2class"
IMAGE_SIZE = 1024
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
NUM_EPOCHS = 100
SEED = 42
USE_AMP = True
RESUME_RUN_ID = None
RUN_HYPERPARAMETER_SEARCH = False
print(dict(MODEL_ID=MODEL_ID, DATASET_TRACK=DATASET_TRACK, IMAGE_SIZE=IMAGE_SIZE, EFFECTIVE_BATCH_SIZE=EFFECTIVE_BATCH_SIZE))

## Dataset validation

Validation checks image existence, dimensions, category IDs, bbox coordinates, zero-area boxes, and class coverage. Statistics expose class counts, size distributions, and objects per image.

In [ ]:
from src.data.validate_annotations import validate_coco
from src.data.statistics import compute_statistics
ann = paths.coco(DATASET_TRACK)/"annotations/instances_train.json"
report = validate_coco(ann, paths.coco(DATASET_TRACK)/"train")
print(report); report.raise_for_errors(); compute_statistics(ann)

## Model construction and introspection

The training command saves architecture, parameter totals, trainable/frozen totals, runtime config, and environment. After a first checkpoint, use notebook 08 for feature shapes, stage strides, FLOPs/MACs, and actual module names.

In [ ]:
YOLOX_ROOT = "/content/YOLOX"
!test -d $YOLOX_ROOT/.git || git clone --depth 1 https://github.com/Megvii-BaseDetection/YOLOX.git $YOLOX_ROOT
%env YOLOX_ROOT=$YOLOX_ROOT
print("The optional adapter is intentionally dependency-gated. Complete the official YOLOX predictor integration before enabling this control.")